<a href="https://colab.research.google.com/github/altaseb12/school/blob/master/politics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os
from urllib.request import urlretrieve
from tqdm import tqdm
import zipfile
import re
import time
from urllib.parse import urljoin

# -----------------------------
# SETTINGS
# -----------------------------
START_URLS = [
    "https://www.bbc.com/amharic",
    "https://www.bbc.com/amharic/topics/cg7265pj1jvt"
]

HEADERS = {"User-Agent": "Mozilla/5.0"}

TARGET_SIZE = 300

BASE_DIR = "bbc_amharic_politics_dataset"
IMG_DIR = os.path.join(BASE_DIR, "images")

os.makedirs(IMG_DIR, exist_ok=True)

data = []
visited = set()
queue = set()

# -----------------------------
# CLEAN TEXT (IMPORTANT 🔥)
# -----------------------------
def clean_text(text):
    text = re.sub(r'Getty Images.*', '', text)
    text = re.sub(r'የፎቶው ባለመብት.*', '', text)
    text = re.sub(r'BBC.*', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# -----------------------------
# IMAGE EXTRACTION
# -----------------------------
def extract_image(soup, base_url):
    img = soup.select_one("figure img")

    if not img:
        img = soup.select_one("img")

    if not img:
        return ""

    if img.get("data-src"):
        url = img["data-src"]
    elif img.get("srcset"):
        url = img["srcset"].split()[0]
    else:
        url = img.get("src", "")

    return urljoin(base_url, url)

# -----------------------------
# POLITICS FILTER
# -----------------------------
def is_politics(text):
    keywords = [
        "ፖለቲካ", "መንግስት", "ምርጫ",
        "ፓርላማ", "ዲፕሎማሲ", "ጦርነት",
        "ህግ", "ስልጣን"
    ]
    return any(k in text for k in keywords)

# -----------------------------
# INITIAL LINK COLLECTION
# -----------------------------
for url in START_URLS:
    res = requests.get(url, headers=HEADERS)
    soup = BeautifulSoup(res.text, "html.parser")

    for a in soup.find_all("a", href=True):
        href = a["href"]
        if "/amharic/articles/" in href:
            queue.add(urljoin("https://www.bbc.com", href))

print("Initial links:", len(queue))

# -----------------------------
# MAIN SCRAPING LOOP
# -----------------------------
while queue and len(data) < TARGET_SIZE:

    link = queue.pop()

    if link in visited:
        continue

    visited.add(link)

    print(f"\n📄 Visiting: {link} | Collected: {len(data)}")

    try:
        res = requests.get(link, headers=HEADERS)

        if res.status_code != 200:
            continue

        soup = BeautifulSoup(res.text, "html.parser")

        # TITLE
        title_tag = soup.find("h1")
        if not title_tag:
            continue

        title = title_tag.get_text(strip=True)

        # TEXT
        paragraphs = soup.find_all("p")
        text = " ".join(p.get_text() for p in paragraphs)
        text = clean_text(text)

        if len(text) < 100:
            continue

        # FILTER POLITICS
        if not is_politics(text):
            continue

        # IMAGE
        img_url = extract_image(soup, link)
        if not img_url:
            continue

        # DOWNLOAD IMAGE
        img_name = f"{len(data)}.jpg"
        img_path = os.path.join(IMG_DIR, img_name)

        try:
            urlretrieve(img_url, img_path)
        except:
            continue

        # SAVE
        data.append({
            "title": title,
            "text": text,
            "category": "politics",
            "image_path": f"images/{img_name}"
        })

        print(f"✅ Saved {len(data)}")

        # -----------------------------
        # 🔥 EXPAND LINKS (SPIDER)
        # -----------------------------
        for a in soup.find_all("a", href=True):
            href = a["href"]
            if "/amharic/articles/" in href:
                full_link = urljoin("https://www.bbc.com", href)

                if full_link not in visited:
                    queue.add(full_link)

        time.sleep(1)

    except Exception as e:
        print("Error:", e)

# -----------------------------
# SAVE CSV
# -----------------------------
df = pd.DataFrame(data)
csv_path = os.path.join(BASE_DIR, "bbc_amharic_politics_news.csv")
df.to_csv(csv_path, index=False, encoding="utf-8-sig")

print("\n✅ TOTAL:", len(data))

# -----------------------------
# ZIP
# -----------------------------
zip_path = "bbc_amharic_politics_dataset.zip"

with zipfile.ZipFile(zip_path, 'w') as z:
    for root, dirs, files in os.walk(BASE_DIR):
        for file in files:
            full = os.path.join(root, file)
            z.write(full, arcname=os.path.relpath(full, BASE_DIR))

print("✅ ZIP READY:", zip_path)

Initial links: 101

📄 Visiting: https://www.bbc.com/amharic/articles/c393dmjn184o | Collected: 0

📄 Visiting: https://www.bbc.com/amharic/articles/cje49q89k29o | Collected: 0

📄 Visiting: https://www.bbc.com/amharic/articles/c203wdn4y26o | Collected: 0

📄 Visiting: https://www.bbc.com/amharic/articles/cqj8egd905zo | Collected: 0

📄 Visiting: https://www.bbc.com/amharic/articles/c5yl74x250eo | Collected: 0

📄 Visiting: https://www.bbc.com/amharic/articles/cjr9y787zplo | Collected: 0

📄 Visiting: https://www.bbc.com/amharic/articles/cre1qyzj3l0o | Collected: 0

📄 Visiting: https://www.bbc.com/amharic/articles/c6ppl536kzro | Collected: 0

📄 Visiting: https://www.bbc.com/amharic/articles/cy7m7mrexxzo | Collected: 0

📄 Visiting: https://www.bbc.com/amharic/articles/c5y8w8w0lgro | Collected: 0
✅ Saved 1

📄 Visiting: https://www.bbc.com/amharic/articles/cn87nv7kz5ko | Collected: 1

📄 Visiting: https://www.bbc.com/amharic/articles/cpqxl45jj1go | Collected: 1

📄 Visiting: https://www.bbc.com/am